In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import psycopg2
from datetime import datetime

# 顯示更完整的輸出
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✓ 導入完成")

In [ ]:
# 資料庫連接設定
# Docker Compose 已將 PostgreSQL 映射到 localhost:5432
DB_USER = "admin"
DB_PASSWORD = "password123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "anime_warehouse"

# 建立連接字串
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(db_url, pool_pre_ping=True)
    
    # 測試連接
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        conn.commit()
    
    print(f"✓ 資料庫連接成功")
    print(f"  主機: {DB_HOST}:{DB_PORT}")
    print(f"  資料庫: {DB_NAME}")
    print(f"  用戶: {DB_USER}")
    
except Exception as e:
    print(f"✗ 連接失敗: {e}")
    print("\n💡 提示:")
    print("  確保 Docker Compose 正在運行: docker compose up -d")
    raise

In [ ]:
# 讀取兩個數據來源

# 1. 讀取乾淨的動畫列表視圖
print("讀取 v_clean_tv_animelist 視圖...")
query_view = """
SELECT * FROM public.v_clean_tv_animelist
LIMIT 5
"""
df_animelist = pd.read_sql_query(query_view, engine)
print(f"✓ 視圖讀取成功: {len(df_animelist)} 筆記錄（顯示前 5 筆）")
print(f"  欄位: {list(df_animelist.columns)}")
print()

# 2. 讀取動畫元數據表
print("讀取 anime_metadata 資料表...")
query_metadata = """
SELECT * FROM public.anime_metadata
LIMIT 5
"""
df_metadata = pd.read_sql_query(query_metadata, engine)
print(f"✓ 表格讀取成功: {len(df_metadata)} 筆記錄（顯示前 5 筆）")
print(f"  欄位: {list(df_metadata.columns)}")
print()

print("=" * 60)
print("完整的統計信息")
print("=" * 60)

# 獲取完整資料筆數
with engine.connect() as conn:
    result_view = conn.execute(text("SELECT COUNT(*) as count FROM public.v_clean_tv_animelist"))
    count_view = result_view.fetchone()[0]
    
    result_metadata = conn.execute(text("SELECT COUNT(*) as count FROM public.anime_metadata"))
    count_metadata = result_metadata.fetchone()[0]
    conn.commit()

print(f"v_clean_tv_animelist 視圖: {count_view:,} 筆動畫")
print(f"anime_metadata 表格: {count_metadata:,} 筆元數據")

In [ ]:
# 加載完整數據集用於機器學習

print("加載完整數據集...")

# 加載 v_clean_tv_animelist 視圖
df_animelist_full = pd.read_sql_query("SELECT * FROM public.v_clean_tv_animelist", engine)
print(f"✓ v_clean_tv_animelist: {df_animelist_full.shape[0]} 筆 × {df_animelist_full.shape[1]} 欄")

# 加載 anime_metadata 表格
df_metadata_full = pd.read_sql_query("SELECT * FROM public.anime_metadata", engine)
print(f"✓ anime_metadata: {df_metadata_full.shape[0]} 筆 × {df_metadata_full.shape[1]} 欄")

# 資料類型
print("\n【v_clean_tv_animelist 數據類型】")
print(df_animelist_full.dtypes)

print("\n【anime_metadata 數據類型】")
print(df_metadata_full.dtypes)

# 缺失值檢查
print("\n【v_clean_tv_animelist 缺失值】")
print(df_animelist_full.isnull().sum())

print("\n【anime_metadata 缺失值】")
print(df_metadata_full.isnull().sum())

In [ ]:
# 數據預覽

print("【v_clean_tv_animelist 前 5 筆】")
print(df_animelist_full.head())

print("\n【anime_metadata 前 5 筆】")
print(df_metadata_full.head())

# 基本統計
print("\n【v_clean_tv_animelist 基本統計】")
print(df_animelist_full.describe(include='all'))